In [1]:
# Clone github repo
!git clone https://github.com/ngvihoa/vi2ch-transliteration.git 

In [2]:
# Used for update repo
%cd /kaggle/working/vi2ch-transliteration/
!git pull
%cd /kaggle/working

In [6]:
# !python3 vi2ch-transliteration/pipelines/thivien_tu_ngon/scripts/crawl_tu_ngon.py --limit 0

In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hoanguen/thivien-dataset/poem.val.csv
/kaggle/input/datasets/hoanguen/thivien-dataset/poem.clean.csv
/kaggle/input/datasets/hoanguen/thivien-dataset/poem.train.csv
/kaggle/input/datasets/hoanguen/thivien-dataset/poem.test.csv


In [7]:
# Load the vi,cn schema produced by the poetry dataset pipeline.

EXPECTED_COLUMNS = ["vi", "cn"]

def load_poem_csv(path):
    frame = pd.read_csv(
        path,
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False,
    )
    if list(frame.columns) != EXPECTED_COLUMNS:
        raise ValueError(
            f"{path}: expected columns {EXPECTED_COLUMNS}, found {list(frame.columns)}"
        )
    if frame.empty:
        raise ValueError(f"{path}: dataset is empty")

    blank_rows = frame[EXPECTED_COLUMNS].apply(lambda column: column.str.strip().eq("")).any(axis=1)
    if blank_rows.any():
        raise ValueError(f"{path}: found {int(blank_rows.sum())} rows with an empty vi or cn value")

    return frame[EXPECTED_COLUMNS].to_dict(orient="records")

In [8]:
# Kiểm tra model
!find /kaggle -type d -name "model"

/kaggle/working/vi2ch-transliteration/model


In [9]:
import os
import sys
import importlib

# 1. Đưa đường dẫn lên đầu danh sách tìm kiếm (ưu tiên cao nhất)
REPO_DIR = '/kaggle/working/vi2ch-transliteration'
if REPO_DIR in sys.path:
    sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Xóa cache module nếu trước đó import lỗi
importlib.invalidate_caches()

# 3. Kiểm tra các file bên trong package model
print("Files trong model/:", os.listdir(os.path.join(REPO_DIR, 'model')))

# 4. Thử import trực tiếp package model
import model
print("-> Import 'model' thành công từ:", model.__file__)

Files trong model/: ['vocabulary.py', '__init__.py', 'configuration_viet_han_bert.py', 'generation.py', 'training.py', 'modeling_viet_han_bert.py', 'data.py', 'tokenization.py']
-> Import 'model' thành công từ: /kaggle/working/vi2ch-transliteration/model/__init__.py


In [10]:
# Cell 1 - Imports + paths

import os
import sys
# Option A: Add the repository path to sys.path
sys.path.append('/kaggle/working/vi2ch-transliteration/model')


import torch
from torch.optim import AdamW
from model import (
    VietHanBertConfig,
    VietHanBertModel,
    VietHanTokenizer,
    build_vocabularies,
    character_bleu,
    create_data_loader,
    evaluate_loss,
    generate_dataset,
    generate_han,
    move_batch_to_device,
    save_checkpoint,
    save_model_bundle,
    save_vocabularies,
    tokenize_han,
    tokenize_vi,
)

# Change INPUT_DIR if the three generated CSV files are attached under another Kaggle dataset path.
INPUT_DIR = "/kaggle/input/datasets/hoanguen/thivien-dataset"
WORK_DIR = "/kaggle/working/vi2ch-transliteration"

TRAIN_PATH = os.path.join(INPUT_DIR, "poem.train.csv")
VAL_PATH = os.path.join(INPUT_DIR, "poem.val.csv")
TEST_PATH = os.path.join(INPUT_DIR, "poem.test.csv")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "checkpoints"), exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print("Device:", DEVICE)
print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)

Device: cuda
Train: /kaggle/input/datasets/hoanguen/thivien-dataset/poem.train.csv
Validation: /kaggle/input/datasets/hoanguen/thivien-dataset/poem.val.csv
Test: /kaggle/input/datasets/hoanguen/thivien-dataset/poem.test.csv


In [11]:
# Cell 2 - Load poem datasets

train_data = load_poem_csv(TRAIN_PATH)
val_data = load_poem_csv(VAL_PATH)
test_data = load_poem_csv(TEST_PATH)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))
print("Sample:", train_data[0])

Train: 19954
Validation: 2493
Test: 2496
Sample: {'vi': 'Bất sinh hồ khoan nhàn chi dã, nhi thác tích ư động thiên.', 'cn': '不生乎寬閒之野，而托跡於洞天。'}


In [12]:
# Cell 3 - Tokenization smoke test

POEM_TEST_VI = "Độc nhiễu hư trai kính,\nThường trì tiểu phủ kha."
POEM_TEST_HAN = "獨繞虛齋徑，\n常持小斧柯。"

print("VI tokens:", tokenize_vi(POEM_TEST_VI))
print("Han tokens:", tokenize_han(POEM_TEST_HAN))

VI tokens: ['Độc', 'nhiễu', 'hư', 'trai', 'kính', ',', 'Thường', 'trì', 'tiểu', 'phủ', 'kha', '.']
Han tokens: ['獨', '繞', '虛', '齋', '徑', '，', '常', '持', '小', '斧', '柯', '。']


In [13]:
# Cell 4 - Build Vietnamese and Han vocabularies from train only

vocab_vi, vocab_han = build_vocabularies(train_data)
VI_VOCAB_PATH, HAN_VOCAB_PATH = save_vocabularies(
    vocab_vi,
    vocab_han,
    os.path.join(WORK_DIR, "vocab"),
)

print("Vietnamese vocab:", len(vocab_vi))
print("Han vocab:", len(vocab_han))
print("Saved:", VI_VOCAB_PATH)
print("Saved:", HAN_VOCAB_PATH)

Vietnamese vocab: 3460
Han vocab: 5601
Saved: /kaggle/working/vi2ch-transliteration/vocab/vocab_vi.json
Saved: /kaggle/working/vi2ch-transliteration/vocab/vocab_han.json


In [14]:
# Cell 5 - Create tokenizer

tokenizer = VietHanTokenizer(VI_VOCAB_PATH, HAN_VOCAB_PATH)

print("VI vocab:", tokenizer.vi_vocab_size)
print("Han vocab:", tokenizer.han_vocab_size)

VI vocab: 3460
Han vocab: 5601


In [15]:
# Cell 6 - Test tokenizer

print("VI tokens:", tokenizer.tokenize_vi(POEM_TEST_VI))
print("VI ids:", tokenizer.encode_vi(POEM_TEST_VI))

han_ids = tokenizer.encode_han(POEM_TEST_HAN)
print("Han tokens:", tokenizer.tokenize_han(POEM_TEST_HAN))
print("Han ids:", han_ids)
print("Han decoded:", tokenizer.decode_han(han_ids))

VI tokens: ['Độc', 'nhiễu', 'hư', 'trai', 'kính', ',', 'Thường', 'trì', 'tiểu', 'phủ', 'kha', '.']
VI ids: [2, 505, 744, 588, 1826, 398, 4, 1384, 173, 317, 433, 1702, 5, 3]
Han tokens: ['獨', '繞', '虛', '齋', '徑', '，', '常', '持', '小', '斧', '柯', '。']
Han ids: [2, 147, 710, 489, 1861, 913, 4, 275, 598, 171, 1928, 1986, 5, 3]
Han decoded: 獨繞虛齋徑，常持小斧柯。


In [16]:
# Cell 7 - Create datasets and loaders

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 16

loader_options = {
    "batch_size": BATCH_SIZE,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "num_workers": 2,
    "pin_memory": True,
}
train_loader = create_data_loader(train_data, tokenizer, shuffle=True, **loader_options)
val_loader = create_data_loader(val_data, tokenizer, **loader_options)
test_loader = create_data_loader(test_data, tokenizer, **loader_options)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 1248
Validation batches: 156
Test batches: 156


In [17]:
# Cell 8 - Inspect one batch

batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)
    print(value[0])

input_ids torch.Size([16, 10])
tensor([  2, 176,  16, 554, 167, 221, 221,  55,   5,   3])
attention_mask torch.Size([16, 10])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
decoder_input_ids torch.Size([16, 9])
tensor([  2, 172, 199, 416, 223, 323, 323,  75,   5])
labels torch.Size([16, 9])
tensor([172, 199, 416, 223, 323, 323,  75,   5,   3])


In [18]:
# Cell 9 - Build model

config = VietHanBertConfig(
    vocab_size=tokenizer.vi_vocab_size,
    hidden_size=512,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=MAX_SOURCE_LENGTH,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    han_vocab_size=tokenizer.han_vocab_size,
    decoder_layers=4,
    decoder_heads=8,
    decoder_ffn_dim=2048,
    max_target_position_embeddings=MAX_TARGET_LENGTH
)

model = VietHanBertModel(config).to(DEVICE)

num_params = sum(
    param.numel()
    for param in model.parameters()
)

print("Parameters:", f"{num_params:,}")

Parameters: 43,371,520


In [19]:
# Cell 10 - Forward-pass smoke test

batch = move_batch_to_device(next(iter(train_loader)), DEVICE)
model.eval()
with torch.no_grad():
    outputs = model(**batch)

print("Loss:", outputs["loss"].item())
print("Logits:", outputs["logits"].shape)

Loss: 8.756119728088379
Logits: torch.Size([16, 9, 5601])


In [20]:
# Cell 11 - Overfit subset of 100 samples

small_loader = create_data_loader(
    train_data[:100],
    tokenizer,
    batch_size=8,
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH,
    shuffle=True,
)
small_model = VietHanBertModel(config).to(DEVICE)
optimizer = AdamW(small_model.parameters(), lr=1e-4, weight_decay=0.0)
small_model.train()

for epoch in range(20):
    total_loss = 0.0
    for batch in small_loader:
        outputs = small_model(**move_batch_to_device(batch, DEVICE))
        loss = outputs["loss"]
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(small_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1:02d} | Loss: {total_loss / len(small_loader):.4f}")

Epoch 01 | Loss: 7.5760
Epoch 02 | Loss: 6.5588
Epoch 03 | Loss: 5.8464
Epoch 04 | Loss: 5.2855
Epoch 05 | Loss: 4.7194
Epoch 06 | Loss: 4.1703
Epoch 07 | Loss: 3.6290
Epoch 08 | Loss: 3.1399
Epoch 09 | Loss: 2.6896
Epoch 10 | Loss: 2.2804
Epoch 11 | Loss: 1.9046
Epoch 12 | Loss: 1.5780
Epoch 13 | Loss: 1.2732
Epoch 14 | Loss: 1.0112
Epoch 15 | Loss: 0.7931
Epoch 16 | Loss: 0.6125
Epoch 17 | Loss: 0.4709
Epoch 18 | Loss: 0.3528
Epoch 19 | Loss: 0.2647
Epoch 20 | Loss: 0.2003


In [22]:
# Cell 12 - Training config

EPOCHS = 5
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM_STEPS = 2

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [23]:
# Cell 13 - Full training

# scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        batch = move_batch_to_device(batch, DEVICE)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs["loss"]
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        if step % 100 == 0:
            print(
                f"Epoch {epoch + 1} | Step {step}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate_loss(model, val_loader, DEVICE)
    print(
        f"\nEpoch {epoch + 1} | Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )
    save_checkpoint(
        model,
        config,
        os.path.join(WORK_DIR, "checkpoints", f"epoch_{epoch + 1}.pt"),
    )

Epoch 1 | Step 0/1248 | Loss 8.8262
Epoch 1 | Step 100/1248 | Loss 6.4763
Epoch 1 | Step 200/1248 | Loss 5.7588
Epoch 1 | Step 300/1248 | Loss 5.5515
Epoch 1 | Step 400/1248 | Loss 6.2845
Epoch 1 | Step 500/1248 | Loss 4.5636
Epoch 1 | Step 600/1248 | Loss 4.4076
Epoch 1 | Step 700/1248 | Loss 3.2592
Epoch 1 | Step 800/1248 | Loss 3.7849
Epoch 1 | Step 900/1248 | Loss 3.4004
Epoch 1 | Step 1000/1248 | Loss 2.9492
Epoch 1 | Step 1100/1248 | Loss 3.0349
Epoch 1 | Step 1200/1248 | Loss 2.2956

Epoch 1 | Train Loss: 4.4635 | Validation Loss: 2.5346
Epoch 2 | Step 0/1248 | Loss 2.5873
Epoch 2 | Step 100/1248 | Loss 2.4858
Epoch 2 | Step 200/1248 | Loss 2.1891
Epoch 2 | Step 300/1248 | Loss 2.0530
Epoch 2 | Step 400/1248 | Loss 2.3985
Epoch 2 | Step 500/1248 | Loss 2.4003
Epoch 2 | Step 600/1248 | Loss 1.6338
Epoch 2 | Step 700/1248 | Loss 2.2313
Epoch 2 | Step 800/1248 | Loss 1.6843
Epoch 2 | Step 900/1248 | Loss 1.4050
Epoch 2 | Step 1000/1248 | Loss 1.4636
Epoch 2 | Step 1100/1248 | Loss 

In [24]:
# Cell 14 - Inference test

model.eval()

prediction = generate_han(
    model,
    tokenizer,
    POEM_TEST_VI,
    DEVICE,
    max_length=MAX_TARGET_LENGTH,
)

print("VI      :", POEM_TEST_VI)
print("EXPECTED:", POEM_TEST_HAN)
print("PREDICT :", prediction)

VI      : Độc nhiễu hư trai kính,
Thường trì tiểu phủ kha.
EXPECTED: 獨繞虛齋徑，
常持小斧柯。
PREDICT : 獨繞虛齋徑，常持小府柯。


In [25]:
# Cell 15 - Test generation on test set

model.eval()

NUM_TEST_SAMPLES = 20

for i, item in enumerate(test_data[:NUM_TEST_SAMPLES]):
    source_text = item["vi"]
    target_text = item["cn"]

    prediction = generate_han(
        model,
        tokenizer,
        source_text,
        DEVICE,
        max_length=MAX_TARGET_LENGTH
    )

    print(f"[{i + 1}]")
    print("VI    :", source_text)
    print("TARGET:", target_text)
    print("PRED  :", prediction)
    print("-" * 80)

[1]
VI    : Vị kiến quân tử,
TARGET: 未見君子，
PRED  : 謂見君子，
--------------------------------------------------------------------------------
[2]
VI    : Mộng hồi phục vật diệc nan tỉnh.
TARGET: 夢回復物亦難醒。
PRED  : 夢回復物亦難省。
--------------------------------------------------------------------------------
[3]
VI    : Phân phân nhất sắc hà do biệt.
TARGET: 紛紛一色何由別。
PRED  : 紛紛一色何由別。
--------------------------------------------------------------------------------
[4]
VI    : Đào hoa kiều khiếp bất thắng phong,
TARGET: 桃花嬌怯不勝風，
PRED  : 桃花橋篋不勝風，
--------------------------------------------------------------------------------
[5]
VI    : Hà Dương chi hội,
TARGET: 河陽之會，
PRED  : 何陽之會，
--------------------------------------------------------------------------------
[6]
VI    : Mục túc liên vân mã đề kiện,
TARGET: 苜蓿連云馬蹄健，
PRED  : 牧足連雲馬啼健，
--------------------------------------------------------------------------------
[7]
VI    : Tam sự tựu tự.
TARGET: 三事就緒。
PRED  : 三事就自。
-------------------------------

In [26]:
# Cell 16 - Full test evaluation

sources, references, predictions = generate_dataset(
    model,
    tokenizer,
    test_data,
    DEVICE,
    max_source_length=MAX_SOURCE_LENGTH,
    max_length=MAX_TARGET_LENGTH,
    log_every=100,
)

print("Finished.")
print("Total samples:", len(predictions))

Processed 100/2496
Processed 200/2496
Processed 300/2496
Processed 400/2496
Processed 500/2496
Processed 600/2496
Processed 700/2496
Processed 800/2496
Processed 900/2496
Processed 1000/2496
Processed 1100/2496
Processed 1200/2496
Processed 1300/2496
Processed 1400/2496
Processed 1500/2496
Processed 1600/2496
Processed 1700/2496
Processed 1800/2496
Processed 1900/2496
Processed 2000/2496
Processed 2100/2496
Processed 2200/2496
Processed 2300/2496
Processed 2400/2496
Finished.
Total samples: 2496


In [27]:
!pip install -q sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.8 MB/s eta 0:00:00


In [28]:
# Cell 18 - Test metrics

char_bleu = character_bleu(predictions, references)
print("=" * 50)
print("CHARACTER-LEVEL BLEU")
print("=" * 50)
print(f"BLEU: {char_bleu.score:.2f}")
print("=" * 50)

CHARACTER-LEVEL BLEU
BLEU: 54.94


In [29]:
for i in range(20):
    print(f"[{i + 1}]")
    print("VI    :", sources[i])
    print("TARGET:", references[i])
    print("PRED  :", predictions[i])
    print("-" * 80)

[1]
VI    : Vị kiến quân tử,
TARGET: 未見君子，
PRED  : 謂見君子，
--------------------------------------------------------------------------------
[2]
VI    : Mộng hồi phục vật diệc nan tỉnh.
TARGET: 夢回復物亦難醒。
PRED  : 夢回復物亦難省。
--------------------------------------------------------------------------------
[3]
VI    : Phân phân nhất sắc hà do biệt.
TARGET: 紛紛一色何由別。
PRED  : 紛紛一色何由別。
--------------------------------------------------------------------------------
[4]
VI    : Đào hoa kiều khiếp bất thắng phong,
TARGET: 桃花嬌怯不勝風，
PRED  : 桃花橋篋不勝風，
--------------------------------------------------------------------------------
[5]
VI    : Hà Dương chi hội,
TARGET: 河陽之會，
PRED  : 何陽之會，
--------------------------------------------------------------------------------
[6]
VI    : Mục túc liên vân mã đề kiện,
TARGET: 苜蓿連云馬蹄健，
PRED  : 牧足連雲馬啼健，
--------------------------------------------------------------------------------
[7]
VI    : Tam sự tựu tự.
TARGET: 三事就緒。
PRED  : 三事就自。
-------------------------------

In [30]:
# Cell 20 - Save final model + tokenizer

FINAL_DIR = save_model_bundle(
    model,
    config,
    vocab_vi,
    vocab_han,
    os.path.join(WORK_DIR, "final"),
)
print("Saved to:", FINAL_DIR)

Saved to: /kaggle/working/vi2ch-transliteration/final


In [4]:
# Cell 21 - Move neccessary files to /public for publish to HF
import json
import torch
from safetensors.torch import save_file

# 1. Save config
config.save_pretrained("./publish/")

# 2. Save weights (safetensors format - được HF ưa thích hơn .bin)
save_file(model.state_dict(), "./publish/model.safetensors")

# 3. Copy vocabulary files
import shutil
shutil.copy("/kaggle/working/vi2ch-transliteration/vocab/vocab_vi.json", "./publish/vocab_vi.json")
shutil.copy("/kaggle/working/vi2ch-transliteration/vocab/vocab_han.json", "./publish/vocab_han.json")

# 4. Copy source files (để trust_remote_code hoạt động)
shutil.copy("/kaggle/working/vi2ch-transliteration/model/configuration_viet_han_bert.py", "./publish/")
shutil.copy("/kaggle/working/vi2ch-transliteration/model/modeling_viet_han_bert.py", "./publish/")
shutil.copy("/kaggle/working/vi2ch-transliteration/model/tokenization.py", "./publish/")


In [36]:
!pip install huggingface_hub safetensors

In [ ]:
# Cell 23 - Publish model to HF
from huggingface_hub import HfApi, login
login(token="hf_xxx")
api = HfApi()
# Tạo repo (nếu chưa có)
api.create_repo("VietHanBERT-vi2cn-v1", private=False)
# Upload toàn bộ thư mục
api.upload_folder(
    folder_path="./publish/",
    repo_id="noah-nguyen-297/VietHanBERT-vi2cn-v1",
    repo_type="model",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/noah-nguyen-297/VietHanBERT-vi2cn-v1/commit/10da3a6a9911120cb85da3471337ff001a675245', commit_message='Upload folder using huggingface_hub', commit_description='', oid='10da3a6a9911120cb85da3471337ff001a675245', pr_url=None, repo_url=RepoUrl('https://huggingface.co/noah-nguyen-297/VietHanBERT-vi2cn-v1', endpoint='https://huggingface.co', repo_type='model', repo_id='noah-nguyen-297/VietHanBERT-vi2cn-v1'), pr_revision=None, pr_num=None)

In [35]:
# Cell 24 - Bundle là download model if needed
import os
from IPython.display import FileLink

# 1. Nén thư mục thành file zip đặt tại /kaggle/working
!zip -r model2.zip /kaggle/working/publish

# 2. Tạo link tải xuống trực tiếp trên giao diện Notebook
FileLink(r'model2.zip')

  adding: kaggle/working/publish/ (stored 0%)
  adding: kaggle/working/publish/config.json (deflated 56%)
  adding: kaggle/working/publish/vocab_vi.json (deflated 67%)
  adding: kaggle/working/publish/vocab_han.json (deflated 66%)
  adding: kaggle/working/publish/configuration_viet_han_bert.py (deflated 58%)
  adding: kaggle/working/publish/modeling_viet_han_bert.py (deflated 71%)
  adding: kaggle/working/publish/tokenization.py (deflated 66%)
  adding: kaggle/working/publish/model.safetensors (deflated 7%)


/kaggle/working/model2.zip

In [44]:
# Cell 25 - Fill test on 8130 pair of "That Ngon Bat Cu" poem
import sys, pandas as pd, torch
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from model.configuration_viet_han_bert import VietHanBertConfig
from model.modeling_viet_han_bert import VietHanBertModel
from model.tokenization import VietHanTokenizer
from model.generation import generate_dataset, character_bleu

REPO_ID = "noah-nguyen-297/VietHanBERT-vi2cn-v1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load tokenizer
vi_vocab  = hf_hub_download(REPO_ID, "vocab_vi.json")
han_vocab = hf_hub_download(REPO_ID, "vocab_han.json")
tokenizer = VietHanTokenizer(vi_vocab, han_vocab)

# 2. Load config & model
config = VietHanBertConfig.from_pretrained(REPO_ID, trust_remote_code=True)
model  = VietHanBertModel(config).to(DEVICE)
weights_path = hf_hub_download(REPO_ID, "model.safetensors")
model.load_state_dict(load_file(weights_path, device=DEVICE))
model.eval()

# 3. Đọc CSV
df = pd.read_csv("vi2ch-transliteration/pipelines/thivien_that_ngon_bat_cu/outputs/that-ngon-bat-cu.csv")
data = df[["vi", "cn"]].dropna().to_dict("records")

# 4. Generate
sources, references, predictions = generate_dataset(
    model, tokenizer, data, DEVICE,
    log_every=100,
)

# 5. BLEU score
bleu = character_bleu(predictions, references)
print(f"Character BLEU: {bleu.score:.2f}")

# 6. Lưu kết quả
out = pd.DataFrame({"vi": sources, "cn_ref": references, "cn_pred": predictions})
out.to_csv("test_results_that_ngon_bat_cu.csv", index=False)
print("Saved to test_results_that_ngon_bat_cu.csv")

# 7. Xem thử vài mẫu
print(out.head(10).to_string())


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Processed 100/8130
Processed 200/8130
Processed 300/8130
Processed 400/8130
Processed 500/8130
Processed 600/8130
Processed 700/8130
Processed 800/8130
Processed 900/8130
Processed 1000/8130
Processed 1100/8130
Processed 1200/8130
Processed 1300/8130
Processed 1400/8130
Processed 1500/8130
Processed 1600/8130
Processed 1700/8130
Processed 1800/8130
Processed 1900/8130
Processed 2000/8130
Processed 2100/8130
Processed 2200/8130
Processed 2300/8130
Processed 2400/8130
Processed 2500/8130
Processed 2600/8130
Processed 2700/8130
Processed 2800/8130
Processed 2900/8130
Processed 3000/8130
Processed 3100/8130
Processed 3200/8130
Processed 3300/8130
Processed 3400/8130
Processed 3500/8130
Processed 3600/8130
Processed 3700/8130
Processed 3800/8130
Processed 3900/8130
Processed 4000/8130
Processed 4100/8130
Processed 4200/8130
Processed 4300/8130
Processed 4400/8130
Processed 4500/8130
Processed 4600/8130
Processed 4700/8130
Processed 4800/8130
Processed 4900/8130
Processed 5000/8130
Processed